# Homework 3

This homework builds upon the above Lab 6, which focuses on developing a GAN model to generate new data points. It aims to deepen the understanding of the GAN model by designing and training the model from scratch. **Follow the steps below to complete the homework.**

- Step 1. Choose an output data modality other than images, such as tabular, textual/language, time series, video, audio, etc. Find a publicly available dataset with a reasonable size (e.g., at least 1K data points). You can always downsample a large dataset to use it for this homework to improve training efficiency.   Please submit your dataset or make the notebook self-contained with it.

- Step 2: Design and train your own GAN model over the above dataset from scratch. Please create your own virtual environment for this homework, and SUBMIT it as a README.txt file for us to reproduce your environment. Please feel free to submit the trained model files to save the execution time of your notebook. If you submit the trained models, your code should handle the DETECTION of existing pre-trained models and LOAD them.

- Step 3: Using the above developed model to generate a new dataset with the same size as your training set. For example, if your original real dataset has N=1,000 data points, you will generate N data points using the developed model.

- Step 4: Evaluate the quality of the generated new dataset D by assessing the model performance. Specifically, you will choose an appropriate model considering your data modality. For example, any ML models can work for tabular data. Transformer or other language models can be chosen for textual data. LSTM can be used for time series. Once you specify the model type, develop two models: model A trained over original real data only, and model B trained over both original real data and generated new data. REPORT IN A MARKDOWN CELL the performance comparison of the two models, A and B, using typical metrics, such as accuracy and F1.

- (Optional Extra 5 Credits) Step 5: This step will be optional, and finishing it will earn 5 extra credits. Enhance the above GAN model (e.g., changing model architecture, polish and enhance training data, hyper-parameter tuning, etc.) so that the model B developed in Step 4 has a higher accuracy (or other metrics) than the model A, i.e., more than >= 10% improvement.


Submit a ZIP file that includes: (1) a Jupyter notebook that shows the above steps. (2) a README.txt to initialize the virtual environment that is required to run your notebook. (3) (optional) your datasets, (4) (optional) the pre-trained models above


The grading of HW 3 focuses on whether you have finished the required steps. You can assume a full credit for HW3 if you finish Steps 1-4 above. Extra credits will be earned once you show the required degree of improvement in Step 5.


In [2]:
import datasets
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dropout, Dense, GRU, Embedding, Reshape, TimeDistributed, BatchNormalization, Input, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Mean
from IPython.core.display import display as jupy_display


In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [4]:
data = datasets.load_dataset("facebook/bouquet", "fra_Latn", split="dev")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

data/sentence_level/dev/fra_Latn.parquet:   0%|          | 0.00/117k [00:00<?, ?B/s]

data/sentence_level/test/fra_Latn.parque(…):   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/504 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/854 [00:00<?, ? examples/s]

In [5]:
def is_french(dataset):
    return dataset['src_lang'] == 'fra_Latn'

In [6]:
data = data.filter(is_french)

source_sentences = data['src_text']
target_sentences = data['tgt_text']

Filter:   0%|          | 0/504 [00:00<?, ? examples/s]

In [7]:
def tokenize(sentences):
    if isinstance(sentences, str):
        return sentences.lower().split()
    elif isinstance(sentences, list):
        return [s.lower().split() for s in sentences]
    else:
        raise TypeError("Input must be a string or a list of strings.")


# tokenized_source = tokenize(source_sentences)
# tokenized_target = tokenize(target_sentences)

In [8]:
print(source_sentences[0])

La fabrication du mahshi varie beaucoup d’un pays à l’autre.


In [9]:
from sklearn.model_selection import train_test_split


text_train, text_dev, fr_train, fr_dev = train_test_split(
    list(source_sentences), list(target_sentences), test_size=0.3, random_state=0)

text_val, text_test, fr_val, fr_test = train_test_split(
    text_dev, fr_dev, test_size=0.3, random_state=0)

In [10]:
PAD, GO, EOS, UNK = START_VOCAB = ['_PAD', '_GO', '_EOS', '_UNK']

In [11]:
# def tokenize(sentence, word_level=True):
#     if word_level:
#         return sentence.split()
#     else:
#         return [sentence[i:i + 1] for i in range(len(sentence))]

In [12]:
def build_vocabulary(tokenized_sequences):
    rev_vocabulary = START_VOCAB[:]
    unique_tokens = set()
    for tokens in tokenized_sequences:
        unique_tokens.update(tokens)
    rev_vocabulary += sorted(unique_tokens)
    vocabulary = {}
    for i, token in enumerate(rev_vocabulary):
        vocabulary[token] = i
    return vocabulary, rev_vocabulary

In [13]:
tokenized_fr_train = tokenize(fr_train)
tokenized_text_train = tokenize(text_train)

fr_vocab, rev_fr_vocab = build_vocabulary(tokenized_fr_train)
text_vocab, rev_text_vocab = build_vocabulary(tokenized_text_train)

In [14]:
len(fr_vocab)

1938

In [15]:
len(text_vocab)

2078

In [16]:
for k, v in sorted(fr_vocab.items())[:10]:
    print(k.rjust(10), v)
print('...')

 "conflict 4
   "dolma" 5
        "i 6
   "yellow 7
       'in 8
     (+/-) 9
     (sop) 10
     (very 11
         - 12
 ...please 13
...


In [17]:
for k, v in sorted(text_vocab.items())[:10]:
    print(k.rjust(10), v)

         ! 4
         % 5
        %. 6
     (+/-) 7
(corned-beef), 8
     (ibi) 9
     (pon) 10
     (très 11
         - 12
       ... 13


In [18]:
print(rev_fr_vocab)

['_PAD', '_GO', '_EOS', '_UNK', '"conflict', '"dolma"', '"i', '"yellow', "'in", '(+/-)', '(sop)', '(very', '-', '...please', '1', '1.', '100', '10:', '11', '12', '12345678.', '12:', '13:', '14,', '14:', '17,000', '18', '1857,', '2', '2%', '2-wheeler', '2.', '2009', '2025.', '28', '2:', '2:>', '3)', '3.', '3.5', '3.start', '30', '30%', '31', '3d', '4)', '4-step', '4-wheeler.', '4.', '5', '5.', '5pm,', '80', '<1:>perfect;', '<2:>love', '<a:>', '<a:>hmm,', '<a:>not', '<a:>what', '<a:>you', "<admin:>we're", '<b:>', '<b:>yea,', "<b:>you're", '<comment', '<customer:>', '<female:>', '<guillermo:>', '<jaime:>', '<male:>', '<student:>', '<technician:>have', '<technician:>i', '<technician:>thank', '<user:>if', '<user:>in', '<yixin:>', '<yong:>', 'a', 'ability', 'able', 'about', 'about;', 'above', 'abrupt,', 'accelerations,', 'accidents', 'accordance', 'according', 'account', 'aches', 'achieved', 'activate', 'active', 'add', 'adjust', 'adjustment:', 'admitted', 'advance', 'advantage', 'advantages

In [19]:
print(rev_text_vocab)

['_PAD', '_GO', '_EOS', '_UNK', '!', '%', '%.', '(+/-)', '(corned-beef),', '(ibi)', '(pon)', '(très', '-', '...', '000', '1', '1.', '10', '100', '11', '12', '12345678.', '13', '1300', '14', '15c3999a.', '17', '18', '1857,', '2', '2.', '2009.', '2025.', '28', '2:>', '3)', '3.', '30', '31', '4', '4)', '4.', '5', '5.', '80', ':', ';', '<1:>', '<2:>', '<a:>', '<a:>ben,', '<a:>qu’est-ce', '<a:>t’es', '<a:>ça', '<admin:>nous', '<b:>', '<b:>ouais,', '<b:>tu', '<cliente:>', '<commentaire', '<femme:>', '<guillermo:>', '<homme:>', '<jaime:>', '<technicien:>je', '<technicien:>merci', '<technicien:>passez', '<user:>au', '<user:>si', '<yixin:>', '<yong:>', '<étudiante:>', '?', 'a', 'a-t-il', 'académiques,', 'accidents', 'accélérations', 'acerbes', 'acheter', 'achetons', 'acheté', 'actif', 'activer', 'activez', 'adapté', 'admis', 'affaires', 'afin', 'agente', 'agençant', 'ah,', 'aider', 'aideront', 'aient', 'aigre-douce', 'aigre-doux', 'aigu', 'ailles.', 'aime', 'aimez', 'ainsi', 'ait', 'ajoutez', '

In [20]:
lens = [len(s) for s in fr_train]

In [21]:
print(max(lens))

302


In [22]:
all_tokenized_sequences = tokenized_fr_train + tokenized_text_train
shared_vocab, rev_shared_vocab = build_vocabulary(all_tokenized_sequences)

In [23]:
def make_input_output(source_tokens, target_tokens, reverse_source=True):
    if reverse_source:
        source_tokens = source_tokens[::-1]
    input_tokens = source_tokens + [GO] + target_tokens
    output_tokens = target_tokens + [EOS]
    return input_tokens, output_tokens


In [24]:
max_length = 302

def vectorize_corpus(source_sequences, target_sequences, shared_vocab,
                     max_length=max_length):
    assert len(source_sequences) == len(target_sequences)
    n_sequences = len(source_sequences)
    source_ids = np.empty(shape=(n_sequences, max_length), dtype=np.int32)
    source_ids.fill(shared_vocab[PAD])
    target_ids = np.empty(shape=(n_sequences, max_length), dtype=np.int32)
    target_ids.fill(shared_vocab[PAD])
    numbered_pairs = zip(range(n_sequences), source_sequences, target_sequences)

    for i, source_seq, target_seq in numbered_pairs:
        source_tokens = tokenize(source_seq)
        target_tokens = tokenize(target_seq)

        in_tokens, out_tokens = make_input_output(source_tokens, target_tokens)

        in_token_ids = [shared_vocab.get(t, shared_vocab[UNK]) for t in in_tokens]
        source_ids[i, -len(in_token_ids):] = in_token_ids

        out_token_ids = [shared_vocab.get(t, shared_vocab[UNK]) for t in out_tokens]
        target_ids[i, -len(out_token_ids):] = out_token_ids
    return source_ids, target_ids

In [25]:
X_train, Y_train = vectorize_corpus(fr_train, text_train, shared_vocab)

In [26]:
X_train.shape

(352, 302)

In [27]:
Y_train.shape

(352, 302)

In [28]:
fr_train[0]

'<B:> Hello, my name is Lisa, a virtual customer service agent and I will be happy to help you with your request, what is the order number please?'

In [29]:
text_train[0]

'<B:> Bonjour, je m’appelle Lisa, je suis une agente de service à la clientèle virtuelle et je serais ravie de vous aider avec votre demande, quel est votre numéro de commande s’il vous plaît ?'

In [30]:
X_val, Y_val = vectorize_corpus(fr_val, text_val, shared_vocab)
X_test, Y_test = vectorize_corpus(fr_test, text_test, shared_vocab)

In [31]:
vocab_size = len(shared_vocab)
simple_seq2seq = Sequential()
simple_seq2seq.add(Embedding(vocab_size, 32))
simple_seq2seq.add(Dropout(0.2))
simple_seq2seq.add(GRU(256, return_sequences=True))
simple_seq2seq.add(Dense(vocab_size, activation='softmax'))

# Here we use the sparse_categorical_crossentropy loss to be able to pass
# integer-coded output for the token ids without having to convert to one-hot codes
simple_seq2seq.compile(optimizer='adam', loss='sparse_categorical_crossentropy')


In [32]:
history = simple_seq2seq.fit(X_train, np.expand_dims(Y_train, -1),
                             validation_data=(X_val, np.expand_dims(Y_val, -1)),
                             epochs=15, verbose=2, batch_size=32)
# We need to use np.expand_dims trick on Y: this is required by Keras because of we use a sparse (integer-based) representation for the output

Epoch 1/15
11/11 - 6s - 553ms/step - loss: 6.5018 - val_loss: 2.1484
Epoch 2/15
11/11 - 1s - 52ms/step - loss: 0.9132 - val_loss: 0.6225
Epoch 3/15
11/11 - 1s - 57ms/step - loss: 0.5573 - val_loss: 0.6372
Epoch 4/15
11/11 - 1s - 56ms/step - loss: 0.5530 - val_loss: 0.6312
Epoch 5/15
11/11 - 1s - 56ms/step - loss: 0.5336 - val_loss: 0.6153
Epoch 6/15
11/11 - 1s - 57ms/step - loss: 0.5111 - val_loss: 0.5967
Epoch 7/15
11/11 - 1s - 53ms/step - loss: 0.4879 - val_loss: 0.5790
Epoch 8/15
11/11 - 1s - 51ms/step - loss: 0.4674 - val_loss: 0.5629
Epoch 9/15
11/11 - 1s - 52ms/step - loss: 0.4491 - val_loss: 0.5488
Epoch 10/15
11/11 - 1s - 52ms/step - loss: 0.4335 - val_loss: 0.5363
Epoch 11/15
11/11 - 1s - 51ms/step - loss: 0.4208 - val_loss: 0.5272
Epoch 12/15
11/11 - 1s - 51ms/step - loss: 0.4091 - val_loss: 0.5179
Epoch 13/15
11/11 - 1s - 51ms/step - loss: 0.3994 - val_loss: 0.5105
Epoch 14/15
11/11 - 1s - 51ms/step - loss: 0.3910 - val_loss: 0.5043
Epoch 15/15
11/11 - 1s - 51ms/step - loss:

In [33]:
latent_dim = 100
embedding_dim = 32
max_length = 302
vocab_size = len(shared_vocab)

def get_generator():
    input_noise = Input(shape=(latent_dim,))

    x = Dense(max_length * embedding_dim)(input_noise)
    x = Reshape((max_length, embedding_dim))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = GRU(256, return_sequences=True)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = GRU(256, return_sequences=True)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = TimeDistributed(Dense(vocab_size, activation='softmax'))(x)

    return Model(inputs=input_noise, outputs=x)


def get_discriminator():
    input_sequence = Input(shape=(max_length,))

    x = Embedding(vocab_size, embedding_dim, input_length=max_length)(input_sequence)

    x = GRU(256)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Dense(1, activation='sigmoid')(x)

    return Model(inputs=input_sequence, outputs=x)

generator = get_generator()
discriminator = get_discriminator()

print("Generator Summary:")
generator.summary()
print("Discriminator Summary:")
discriminator.summary()

Generator Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9664)           │       976,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 302, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 302, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 302, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 302, 256)       │       222,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 302, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 302, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 302, 256)       │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 302, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 302, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 302, 3815)      │       980,455 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,576,167 (9.83 MB)

...

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 302)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 302, 32)        │       122,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 256)            │       222,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 346,081 (1.32 MB)

 Trainable params: 345,569 (1.32 MB)

 Non-trainable params: 512 (2.00 KB)

In [34]:
def get_noise(batch_size, nz=latent_dim):
    return tf.random.normal([batch_size, nz])

noise = get_noise(20)

print("init", noise.shape)
fake_sequences_probs = generator(noise)
print("Fake sequence probabilities", fake_sequences_probs.shape)

# To pass to discriminator, we need to convert probabilities to token IDs.
# For a sanity check, we can take the argmax to get the most likely token ID for each position.
fake_sequences_ids = tf.argmax(fake_sequences_probs, axis=-1, output_type=tf.int32)
print("Fake sequence IDs for discriminator input", fake_sequences_ids.shape)

preds = discriminator(fake_sequences_ids)
print("Predictions", preds.shape)

init (20, 100)
Fake sequence probabilities (20, 302, 3815)
Fake sequence IDs for discriminator input (20, 302)
Predictions (20, 1)


In [35]:
def discriminator_loss(preds_real, preds_fake):
    # from_logits=False because the discriminator now has a sigmoid activation
    loss_real = binary_crossentropy(tf.ones_like(preds_real), preds_real, from_logits=False)
    loss_fake = binary_crossentropy(tf.zeros_like(preds_fake), preds_fake, from_logits=False)
    return loss_real + loss_fake


def generator_loss(preds_fake):
    # from_logits=False because the discriminator now has a sigmoid activation
    return binary_crossentropy(tf.ones_like(preds_fake), preds_fake, from_logits=False)

In [36]:
optimizer_d = Adam(learning_rate=1e-4, beta_1=0.5)
optimizer_g = Adam(learning_rate=1e-4, beta_1=0.5)

In [37]:
@tf.function
def train_step(images):
    noise = get_noise(images.shape[0])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
      generated_sequences_probs = generator(noise, training=True)

      real_output = discriminator(images, training=True)

      generated_sequences_ids_for_disc = tf.argmax(generated_sequences_probs, axis=-1, output_type=tf.int32)
      fake_output_for_disc = discriminator(generated_sequences_ids_for_disc, training=True)

      disc_loss = discriminator_loss(real_output, fake_output_for_disc)

      embedding_layer_weights = discriminator.layers[1].embeddings

      soft_generated_embeddings = tf.matmul(generated_sequences_probs, embedding_layer_weights)

      fake_output_for_gen = discriminator_body_for_gen(soft_generated_embeddings, training=True)

      gen_loss = generator_loss(fake_output_for_gen)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    optimizer_g.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    optimizer_d.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return disc_loss, gen_loss

def get_discriminator_body_model(original_discriminator):
    input_embeddings = Input(shape=(max_length, embedding_dim), dtype=tf.float32)

    x = original_discriminator.layers[2](input_embeddings)
    x = original_discriminator.layers[3](x)
    x = original_discriminator.layers[4](x)
    x = original_discriminator.layers[5](x)
    return Model(inputs=input_embeddings, outputs=x)

discriminator_body_for_gen = get_discriminator_body_model(discriminator)

In [38]:
def decode_sequences_and_display(sequences_probs, rev_vocab, num_display=5):
    fake_sequences_ids = tf.argmax(sequences_probs, axis=-1, output_type=tf.int32).numpy()
    print("\nGenerated Sequences:")
    for i in range(min(num_display, fake_sequences_ids.shape[0])):

        words = [rev_vocab[idx] for idx in fake_sequences_ids[i] if idx != shared_vocab[PAD] and idx != shared_vocab[EOS]]

        text = ' '.join(words).replace(GO, '').strip()
        print(f"  {i+1}: {text}")
    print("\n")

epochs = 100

fixed_noise = get_noise(20)

print("Base noise:")
fake_texts_probs = generator(fixed_noise, training=False).numpy()
decode_sequences_and_display(fake_texts_probs, rev_shared_vocab)

for epoch in range(epochs):
    print("====== Epoch {:2d} ======".format(epoch))

    epoch_loss_d = Mean()
    epoch_loss_g = Mean()

    num_batches = X_train.shape[0] // 100
    for i in range(num_batches):
        start_idx = i * 100
        end_idx = (i + 1) * 100
        real_text_batch = X_train[start_idx:end_idx]

        loss_d, loss_g = train_step(real_text_batch)
        epoch_loss_d(loss_d)
        epoch_loss_g(loss_g)

        if i % 50 == 0 and i > 0:
            print(i, end=" ... ")

    print("Discriminator: {}, Generator: {}".format(
        epoch_loss_d.result(), epoch_loss_g.result()))
    fake_texts_probs = generator(fixed_noise, training=False).numpy()
    decode_sequences_and_display(fake_texts_probs, rev_shared_vocab)

Base noise:

Generated Sequences:
  1: euros doivent doivent doivent doivent doivent envers envers raison. raison. s’opposent doivent doivent doivent doivent doivent doivent doivent doivent doivent doivent euros doivent euros euros language. euros euros doivent everyone! everyone! doivent doivent doivent doivent doivent doivent doivent doivent be, ils ils l’essayer l’homme. partners partners euros doivent doivent doivent doivent mis commencé here's here's heures heures motivant doivent euros euros différents doivent différents doivent share doivent doivent doivent doivent doivent doivent doivent doivent doivent doivent doivent doivent fonctionnaire, doivent doivent doivent eyes, doivent doivent partners share doivent doivent doivent doivent doivent doivent ingredients: share dettes doivent doivent partners lavez lavez doivent opérationnelles doivent doivent doivent doivent doivent doivent wheels? doivent doivent doivent euros doivent doivent doivent doivent doivent doivent doivent derr

In [39]:
N_train = X_train.shape[0]
batch_size_gen = 20

all_generated_sequences_ids = []


for i in range(0, N_train, batch_size_gen):
    current_batch_size = min(batch_size_gen, N_train - i)
    noise_batch = get_noise(current_batch_size)
    fake_sequences_probs_batch = generator(noise_batch, training=False)
    fake_sequences_ids_batch = tf.argmax(fake_sequences_probs_batch, axis=-1, output_type=tf.int32)
    all_generated_sequences_ids.append(fake_sequences_ids_batch.numpy())

Y_gen_ids = np.concatenate(all_generated_sequences_ids, axis=0)

random_indices = np.random.choice(N_train, N_train, replace=True)
X_gen_ids = X_train[random_indices]

print(f"Shape of generated Y (Y_gen_ids): {Y_gen_ids.shape}")
print(f"Shape of sampled X (X_gen_ids): {X_gen_ids.shape}")

X_combined_train = np.concatenate([X_train, X_gen_ids], axis=0)
Y_combined_train = np.concatenate([Y_train, Y_gen_ids], axis=0)

print(f"Shape of combined X for Model B: {X_combined_train.shape}")
print(f"Shape of combined Y for Model B: {Y_combined_train.shape}")

Shape of generated Y (Y_gen_ids): (352, 302)
Shape of sampled X (X_gen_ids): (352, 302)
Shape of combined X for Model B: (704, 302)
Shape of combined Y for Model B: (704, 302)


In [40]:
def create_seq2seq_model(vocab_size, embedding_dim, max_length):
    model = Sequential()
    model.add(Embedding(vocab_size, embedding_dim, input_length=max_length))
    model.add(Dropout(0.2))
    model.add(GRU(256, return_sequences=True))
    model.add(Dense(vocab_size, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

vocab_size = len(shared_vocab)
embedding_dim = 32

print("Training Model A (Real Data Only)...")
model_A = create_seq2seq_model(vocab_size, embedding_dim, max_length)
history_A = model_A.fit(X_train, np.expand_dims(Y_train, -1),
                        validation_data=(X_val, np.expand_dims(Y_val, -1)),
                        epochs=30, verbose=0, batch_size=32)

print("Training Model B (Real + Generated Data)...")
model_B = create_seq2seq_model(vocab_size, embedding_dim, max_length)
history_B = model_B.fit(X_combined_train, np.expand_dims(Y_combined_train, -1),
                        validation_data=(X_val, np.expand_dims(Y_val, -1)),
                        epochs=30, verbose=0, batch_size=32)

print("Training complete for both models.")

Training Model A (Real Data Only)...
Training Model B (Real + Generated Data)...
Training complete for both models.


In [41]:
print("Evaluating Model A")
loss_A, accuracy_A = model_A.evaluate(X_test, np.expand_dims(Y_test, -1), verbose=0)
print(f"Model A (Real Data Only) Test Loss: {loss_A:.4f}")
print(f"Model A (Real Data Only) Test Accuracy (token-level): {accuracy_A:.4f}")

print("Evaluating Model B")
loss_B, accuracy_B = model_B.evaluate(X_test, np.expand_dims(Y_test, -1), verbose=0)
print(f"Model B (Real + Generated Data) Test Loss: {loss_B:.4f}")
print(f"Model B (Real + Generated Data) Test Accuracy (token-level): {accuracy_B:.4f}")


Evaluating Model A
Model A (Real Data Only) Test Loss: 0.3580
Model A (Real Data Only) Test Accuracy (token-level): 0.9564
Evaluating Model B
Model B (Real + Generated Data) Test Loss: 0.8701
Model B (Real + Generated Data) Test Accuracy (token-level): 0.9498


Adding the GAN-generated data does not necesarily improve performance in this scenario. We may need to evaluate options to generate better quality data somehow.